In [1]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from scipy.stats import mannwhitneyu
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

# ---- CHANGE THESE PATHS ----
DATA_PATH = r"C:\Users\minsa_\Downloads\Final_Features_MultiLabel.csv"
IMAGE_DIR = r"C:\Users\minsa_\OneDrive\Desktop\Disease_Cell_Images"
OUT_DIR   = r"C:\Users\minsa_\Desktop\Processed_SP_ML"     

OUT_W, OUT_H = 512, 384
RANDOM_STATE = 42
LABEL_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]
FEATURES = ["ECD_CD_cells_per_mm2", "ECD_Deficit", "SD", "CV", "HEX_percent", "Donor_Age"]
os.makedirs(os.path.join(OUT_DIR, "images"), exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Labelled rows:", len(df))

Labelled rows: 1048


In [2]:
# rebuilt exactly as in 02d (same rows, key and seed), so image and numbers-only models use IDENTICAL folds
train_all = df[df["Split"] == "train"].reset_index(drop=True)
strat_key = train_all["FECD"] * 4 + train_all["Guttata"] * 2 + train_all["Folds"]
fold_map = {}
for k, (_, va) in enumerate(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
                            .split(train_all, strat_key, groups=train_all["Donor_Group"])):
    for tid in train_all.loc[va, "Tissue_ID"]:
        fold_map[tid] = k
df["CV_Fold"] = df["Tissue_ID"].map(fold_map).fillna(-1).astype(int)     # -1 = test row
pd.crosstab(df["CV_Fold"], df["Split"])

Split,test,train
CV_Fold,,
-1,210,0
0,0,168
1,0,168
2,0,167
3,0,168
4,0,167


In [3]:
def tissue_key(s):
    m = re.search(r"(\d{4,6})\s*([LR])", str(s))
    return (int(m.group(1)), m.group(2)) if m else None

files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith("_sp.jpg")]
img_index = pd.DataFrame({"SP_File": files})
img_index["Key"] = img_index["SP_File"].map(tissue_key)
print("SP images in folder:", len(img_index), "| unparsed names:", img_index["Key"].isna().sum(),
      "| duplicate tissue+eye:", img_index["Key"].duplicated().sum())
img_index = img_index.dropna(subset=["Key"]).drop_duplicates("Key")

df["Key"] = df["Tissue_ID"].map(tissue_key)
m = df.merge(img_index, on="Key", how="left")
m["Has_Image"] = m["SP_File"].notna()
print("Rows with an image:", m["Has_Image"].sum(), "of", len(m))
display(pd.DataFrame({"With image": m.loc[m.Has_Image, LABEL_COLS + ["Normal"]].sum(),
                      "Without image": m.loc[~m.Has_Image, LABEL_COLS + ["Normal"]].sum()}))
print("Labelled rows with no image:")
m.loc[~m["Has_Image"], ["Tissue_ID", "Combination", "Split"]]

SP images in folder: 1042 | unparsed names: 0 | duplicate tissue+eye: 0
Rows with an image: 1033 of 1048


,With image,Without image
FECD,126,1
Guttata,100,0
Folds,380,0
PolyPleo,131,1
LowCD,125,1
Normal,490,14


Labelled rows with no image:


,Tissue_ID,Combination,Split
29,NEB-2020-09-12683 L,Normal,test
70,NEB-2020-10-12726 R,Normal,train
77,NEB-2020-10-12733 L,Normal,train
186,NEB-2022-10-14315 L,Normal,train
245,NEB-2023-02-14710 R,FECD + PolyPleo + LowCD,train
383,NEB-2023-04-14996 R,Normal,train
527,NEB-2023-08-15300 R,Normal,train
742,NEB-2024-06-16116 L,Normal,train
743,NEB-2024-06-16117 R,Normal,train
756,NEB-2024-06-16130 L,Normal,train


In [4]:
def read_gray(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if g is None:
        raise ValueError(f"Cannot read {path}")
    return g

def quality_metrics(path):
    g = cv2.resize(read_gray(path), (640, 480), interpolation=cv2.INTER_AREA)
    return {"Q_Mean": g.mean(), "Q_Std": g.std(), "Q_Focus": cv2.Laplacian(g, cv2.CV_64F).var(),
            "Q_Dark": (g < 20).mean(), "Q_Saturated": (g > 245).mean()}

img = m[m["Has_Image"]].copy()
img = img.join(pd.DataFrame([quality_metrics(os.path.join(IMAGE_DIR, f)) for f in img["SP_File"]],
                            index=img.index))

FOCUS_LOW = img["Q_Focus"].quantile(0.02)      # blurriest 2%
DARK_HIGH = 0.45
def quality_flag(r):
    if r["Q_Std"] < 5 or r["Q_Saturated"] > 0.5:
        return "Unusable"                           # blank / white capture -> excluded
    if r["Q_Focus"] < FOCUS_LOW or r["Q_Dark"] > DARK_HIGH:
        return "Low"                                # kept, but flagged
    return "OK"

img["Quality"] = img.apply(quality_flag, axis=1)
print(f"Focus threshold: {FOCUS_LOW:.2f} | dark threshold: {DARK_HIGH}")
display(img["Quality"].value_counts())
img.loc[img["Quality"] == "Unusable", ["Tissue_ID", "Combination", "Q_Mean", "Q_Std"]]

Focus threshold: 23.11 | dark threshold: 0.45


Quality
OK          1011
Low           21
Unusable       1
Name: count, dtype: int64

,Tissue_ID,Combination,Q_Mean,Q_Std
955,NEB-2025-10-17551 L,Normal,255.0,0.0
